# Train LRRo Word Classifiers (Attention Pooling + MLP)

Trains the attention-pooling + MLP heads used for word-level
classification on LRRo, on top of frozen VSR encoder representations.

Produces two checkpoints (one for the LAB split, one for the WILD
split) for the chosen preprocessing strategy. To produce the four
strategies reported in the paper, change `STRATEGY` and re-run.

Pipeline:
1. Download LRRo from Zenodo and extract
2. Load the frozen VTP visual encoder + VSR transformer encoder
3. Extract sequence embeddings for each LRRo clip (one strategy)
4. Train two MLP heads — one on Lab train, one on Wild train
5. Evaluate on the held-out test sets and save the checkpoints

In [ ]:
# Setup — clone MultiVSR (for the model code) and install deps
!git clone https://github.com/Sindhu-Hegde/multivsr.git
%cd multivsr
!pip install -q -r requirements.txt
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q huggingface_hub tqdm pandas numpy scikit-learn pillow

# Download VTP visual encoder checkpoint
!mkdir -p checkpoints
!wget -q -O checkpoints/feature_extractor.pth \
    "https://www.robots.ox.ac.uk/~vgg/research/vtp-for-lip-reading/checkpoints/extended_train_data/feature_extractor.pth"

In [ ]:
# Imports
import sys, os, glob, random
sys.path.append("/content/multivsr")
sys.argv = ['']

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from tqdm import tqdm

from models import build_model, build_visual_encoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Configuration

# Choose preprocessing strategy: "64_bottom", "64_middle", "96_resize"
STRATEGY = "64_bottom"

# Pretrained VSR encoder to use for extracting embeddings
MODEL_REPO = "vsro200/models-vsro200"
MODEL_CKPT = "checkpoints/model_200h_auto.pt"

# MLP hyperparameters
HIDDEN_DIM = 512
DROPOUT = 0.5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 100  # No aggressive early stopping

# Where to save the two checkpoints (best_lab_clf.pt and best_wild_clf.pt)
OUTPUT_DIR = f"./checkpoints_{STRATEGY}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Strategy: {STRATEGY}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Download LRRo dataset from Zenodo
import subprocess
import tarfile

LRRO_URL = "https://zenodo.org/records/3753559/files/LRRo_data_set.tar.gz"
LRRO_DIR = "/content/lrro"
MAIN_ARCHIVE = "/content/lrro_main.tar.gz"

if not os.path.exists(LRRO_DIR):
    print("Downloading LRRo ...")
    !curl -L -o {MAIN_ARCHIVE} "{LRRO_URL}"
    !mkdir -p {LRRO_DIR}

# The Zenodo file has a .tar.gz extension but is actually a RAR archive
print("Inspecting the archive:")
!file {MAIN_ARCHIVE}

print("\nExtracting main archive (using unrar) ...")
!unrar x -o+ {MAIN_ARCHIVE} {LRRO_DIR}/ > /dev/null

# Extract sub-archives if they exist
sub_archives = glob.glob(os.path.join(LRRO_DIR, "**/*.tar.gz"), recursive=True) + \
               glob.glob(os.path.join(LRRO_DIR, "**/*.rar"), recursive=True)

if sub_archives:
    print(f"\nFound {len(sub_archives)} sub-archives, extracting ...")
    for sub in sub_archives:
        print(f"  Extracting {os.path.basename(sub)} ...")
        dest = os.path.dirname(sub)
        is_rar = subprocess.run(["unrar", "t", sub], capture_output=True).returncode == 0
        if is_rar:
            subprocess.run(["unrar", "x", "-o+", sub, dest + "/"], capture_output=True)
        else:
            try:
                with tarfile.open(sub, 'r:*') as tar:
                    tar.extractall(path=dest)
            except Exception as e:
                print(f"    Error: {e}")

print("\nLRRo extracted successfully.")

In [ ]:
# Load the frozen VTP visual encoder and the VSR transformer encoder

# VTP visual encoder (original, frozen)
visual_encoder = build_visual_encoder().to(device)
s = torch.load("/content/multivsr/checkpoints/feature_extractor.pth",
               map_location=device)["state_dict"]
new_s = {k.replace("module.face_encoder.", ""): v
         for k, v in s.items() if "face_encoder" in k}
visual_encoder.load_state_dict(new_s)
visual_encoder.eval()
for p in visual_encoder.parameters():
    p.requires_grad = False
print("VTP visual encoder loaded")

# VSR encoder-decoder (frozen — we only use the encoder for representations)
model = build_model().to(device)
lm_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_CKPT,
                           repo_type="model")
model.load_state_dict(torch.load(lm_path, map_location=device))
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"VSR encoder loaded from {MODEL_REPO}/{MODEL_CKPT}")

In [ ]:
# Four preprocessing strategies for placing 64x64 LRRo crops on a 96x96 canvas

CANVAS_SIZE = 96
GRAY_PAD_VALUE = 0.5


def _list_jpgs(clip_dir):
    return sorted(glob.glob(os.path.join(clip_dir, "*.jpg")),
                  key=lambda x: int(os.path.splitext(os.path.basename(x))[0]))


def load_lrro_clip_64_bottom(clip_dir):
    """Keep 64x64, place center-bottom on 96x96 gray canvas."""
    img_size = 64
    x_off = (CANVAS_SIZE - img_size) // 2
    y_off = CANVAS_SIZE - img_size
    jpgs = _list_jpgs(clip_dir)
    if not jpgs:
        return None
    frames = []
    for p in jpgs:
        img = Image.open(p).convert('L')
        arr = np.array(img, dtype=np.float32) / 255.0
        canvas = np.full((CANVAS_SIZE, CANVAS_SIZE), GRAY_PAD_VALUE, dtype=np.float32)
        canvas[y_off:y_off + img_size, x_off:x_off + img_size] = arr
        frames.append(np.stack([canvas, canvas, canvas], axis=0))
    video = np.stack(frames, axis=0)
    return np.transpose(video, (1, 0, 2, 3))


def load_lrro_clip_64_middle(clip_dir):
    """Keep 64x64, place exactly center on 96x96 gray canvas."""
    img_size = 64
    x_off = (CANVAS_SIZE - img_size) // 2
    y_off = (CANVAS_SIZE - img_size) // 2
    jpgs = _list_jpgs(clip_dir)
    if not jpgs:
        return None
    frames = []
    for p in jpgs:
        img = Image.open(p).convert('L')
        arr = np.array(img, dtype=np.float32) / 255.0
        canvas = np.full((CANVAS_SIZE, CANVAS_SIZE), GRAY_PAD_VALUE, dtype=np.float32)
        canvas[y_off:y_off + img_size, x_off:x_off + img_size] = arr
        frames.append(np.stack([canvas, canvas, canvas], axis=0))
    video = np.stack(frames, axis=0)
    return np.transpose(video, (1, 0, 2, 3))


def load_lrro_clip_96_resize(clip_dir):
    """Resize 64x64 directly to 96x96 (no padding)."""
    jpgs = _list_jpgs(clip_dir)
    if not jpgs:
        return None
    frames = []
    for p in jpgs:
        img = Image.open(p).convert('L').resize((CANVAS_SIZE, CANVAS_SIZE), Image.BICUBIC)
        arr = np.array(img, dtype=np.float32) / 255.0
        frames.append(np.stack([arr, arr, arr], axis=0))
    video = np.stack(frames, axis=0)
    return np.transpose(video, (1, 0, 2, 3))


PREPROCESSING_FNS = {
    "64_bottom": load_lrro_clip_64_bottom,
    "64_middle": load_lrro_clip_64_middle,
    "96_resize": load_lrro_clip_96_resize,
}

load_lrro_clip = PREPROCESSING_FNS[STRATEGY]
print(f"Selected preprocessing: {STRATEGY}")

In [ ]:
# Scan LRRo and build the index of clips

def scan_lrro(base_dir):
    records = []
    if os.path.exists(os.path.join(base_dir, "LRRo data set")):
        base_dir = os.path.join(base_dir, "LRRo data set")

    for dataset_name in ['Lab_LRRo_data_set', 'Wild_LRRo_data_set']:
        dataset_dir = os.path.join(base_dir, dataset_name)
        if not os.path.isdir(dataset_dir):
            continue
        ds_label = 'Lab' if 'Lab' in dataset_name else 'Wild'

        for split in ['train', 'val', 'test']:
            split_dir = os.path.join(dataset_dir, split)
            if not os.path.isdir(split_dir):
                continue
            for word_name in sorted(os.listdir(split_dir)):
                word_dir = os.path.join(split_dir, word_name)
                if not os.path.isdir(word_dir):
                    continue
                for clip_name in sorted(os.listdir(word_dir)):
                    clip_dir = os.path.join(word_dir, clip_name)
                    if not os.path.isdir(clip_dir):
                        continue
                    n_jpgs = len([f for f in os.listdir(clip_dir) if f.endswith('.jpg')])
                    if n_jpgs > 0:
                        records.append({
                            'clip_dir': clip_dir,
                            'word': word_name,
                            'split': split,
                            'dataset': ds_label,
                            'n_frames': n_jpgs,
                        })
    return pd.DataFrame(records)

lrro_df = scan_lrro(LRRO_DIR)
print(f"Total clips: {len(lrro_df)}")
lrro_df.head()

In [ ]:
# Build separate class mappings for Lab and Wild

lab_df = lrro_df[lrro_df['dataset'] == 'Lab']
wild_df = lrro_df[lrro_df['dataset'] == 'Wild']

lab_words = sorted(lab_df['word'].unique())
lab_class_to_idx = {w: i for i, w in enumerate(lab_words)}
lab_idx_to_class = {i: w for w, i in lab_class_to_idx.items()}
N_CLASSES_LAB = len(lab_class_to_idx)

wild_words = sorted(wild_df['word'].unique())
wild_class_to_idx = {w: i for i, w in enumerate(wild_words)}
wild_idx_to_class = {i: w for w, i in wild_class_to_idx.items()}
N_CLASSES_WILD = len(wild_class_to_idx)

print(f"LAB classes:  {N_CLASSES_LAB}")
print(f"WILD classes: {N_CLASSES_WILD}")

In [ ]:
# Extract sequence embeddings for all clips (train, val, test, separately for Lab and Wild)

def extract_embeddings(df, label_name, class_mapping):
    sequences, labels = [], []
    errors = 0
    print(f"Extracting {label_name}: {len(df)} clips ...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        video = load_lrro_clip(row['clip_dir'])
        if video is None:
            errors += 1
            continue

        vt = torch.FloatTensor(video).unsqueeze(0).to(device)

        with torch.no_grad(), torch.amp.autocast('cuda'):
            vid_emb = visual_encoder(vt)
            B, T_enc, D = vid_emb.size()
            src_mask = torch.ones((B, 1, T_enc), device=device).bool()
            memory, _ = model.encode(vid_emb, src_mask)

        sequences.append(memory.squeeze(0).float().cpu().numpy())
        labels.append(class_mapping[row['word']])
        del vt

    print(f"  Extracted: {len(sequences)}, errors: {errors}")
    return sequences, labels


# LAB splits
train_lab_df = lab_df[lab_df['split'] == 'train']
val_lab_df = lab_df[lab_df['split'] == 'val']
test_lab_df = lab_df[lab_df['split'] == 'test']

print("\n--- LAB ---")
train_lab_seqs, train_lab_labels = extract_embeddings(train_lab_df, "Train Lab", lab_class_to_idx)
val_lab_seqs, val_lab_labels = extract_embeddings(val_lab_df, "Val Lab", lab_class_to_idx)
test_lab_seqs, test_lab_labels = extract_embeddings(test_lab_df, "Test Lab", lab_class_to_idx)

# WILD splits
train_wild_df = wild_df[wild_df['split'] == 'train']
val_wild_df = wild_df[wild_df['split'] == 'val']
test_wild_df = wild_df[wild_df['split'] == 'test']

print("\n--- WILD ---")
train_wild_seqs, train_wild_labels = extract_embeddings(train_wild_df, "Train Wild", wild_class_to_idx)
val_wild_seqs, val_wild_labels = extract_embeddings(val_wild_df, "Val Wild", wild_class_to_idx)
test_wild_seqs, test_wild_labels = extract_embeddings(test_wild_df, "Test Wild", wild_class_to_idx)

EMB_DIM = train_lab_seqs[0].shape[1]
print(f"\nEMB_DIM = {EMB_DIM}")

In [ ]:
# Dataset + model definitions

class WordSeqDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), self.labels[idx]


def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = [s.size(0) for s in seqs]
    max_len = max(lengths)
    D = seqs[0].size(1)
    padded = torch.zeros(len(seqs), max_len, D)
    mask = torch.zeros(len(seqs), 1, max_len).bool()
    for i, s in enumerate(seqs):
        end = lengths[i]
        padded[i, :end] = s
        mask[i, 0, :end] = True
    return padded, mask, torch.LongTensor(labels)


class AttentionPooling(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 1),
        )
    def forward(self, x, mask=None):
        weights = self.attn(x).squeeze(-1)
        if mask is not None:
            weights.masked_fill_(~mask.squeeze(1), -1e9)
        attn_weights = torch.softmax(weights, dim=-1)
        pooled = torch.bmm(attn_weights.unsqueeze(1), x).squeeze(1)
        return pooled, attn_weights


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.pool = AttentionPooling(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x, mask=None):
        pooled, attn = self.pool(x, mask)
        return self.net(pooled), attn


def evaluate_model(clf, loader, num_classes):
    clf.eval()
    c1 = c5 = total = 0
    with torch.no_grad():
        for seqs, mask, labels in loader:
            seqs, mask, labels = seqs.to(device), mask.to(device), labels.to(device)
            logits, _ = clf(seqs, mask)
            c1 += (logits.argmax(1) == labels).sum().item()
            top_k = min(5, num_classes)
            c5 += (logits.topk(top_k, dim=1).indices == labels.unsqueeze(1)).any(1).sum().item()
            total += labels.size(0)
    return c1 / total * 100, c5 / total * 100

In [ ]:
# Train MLPs separately for Lab and Wild

def train_mlp_model(name, train_seqs, train_labels, val_seqs, val_labels,
                     test_seqs, test_labels, num_classes):
    print(f"\n{'='*60}")
    print(f"Training: {name} ({num_classes} classes)")
    print(f"{'='*60}")

    train_loader = DataLoader(WordSeqDataset(train_seqs, train_labels),
                              batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(WordSeqDataset(val_seqs, val_labels),
                            batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(WordSeqDataset(test_seqs, test_labels),
                             batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    clf = MLP(EMB_DIM, HIDDEN_DIM, num_classes, DROPOUT).to(device)
    optimizer = torch.optim.Adam(clf.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc1 = 0
    best_val_acc5 = 0
    no_improve = 0
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{name.lower()}_clf.pt")

    print(f"{'Ep':>4s} {'Loss':>7s} {'Tr%':>5s} {'Val1%':>6s} {'Val5%':>6s}")
    print('-' * 40)

    for epoch in range(EPOCHS):
        clf.train()
        tloss = tcorr = ttot = 0
        for seqs, mask, labels in train_loader:
            seqs, mask, labels = seqs.to(device), mask.to(device), labels.to(device)
            logits, _ = clf(seqs, mask)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tloss += loss.item() * labels.size(0)
            tcorr += (logits.argmax(1) == labels).sum().item()
            ttot += labels.size(0)

        tacc = tcorr / ttot * 100
        val1, val5 = evaluate_model(clf, val_loader, num_classes)
        scheduler.step(val1)

        if val1 > best_val_acc1:
            best_val_acc1, best_val_acc5 = val1, val5
            no_improve = 0
            torch.save(clf.state_dict(), best_model_path)
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0 or epoch == 0 or no_improve == 0:
            mark = '*' if no_improve == 0 else ' '
            print(f"{epoch+1:4d} {tloss/ttot:7.4f} {tacc:5.1f} {val1:6.1f} {val5:6.1f} {mark}")

        if no_improve >= PATIENCE:
            print(f"Early stop @ {epoch + 1}")
            break

    print(f"\nBest validation: Top1 = {best_val_acc1:.1f}%")

    # Final evaluation on test set
    clf.load_state_dict(torch.load(best_model_path))
    test_top1, test_top5 = evaluate_model(clf, test_loader, num_classes)
    print(f"TEST {name}: Top1 = {test_top1:.1f}% | Top5 = {test_top5:.1f}%")
    return clf, test_top1, test_top5


clf_lab, lab_test_top1, lab_test_top5 = train_mlp_model(
    'Lab',
    train_lab_seqs, train_lab_labels,
    val_lab_seqs, val_lab_labels,
    test_lab_seqs, test_lab_labels,
    N_CLASSES_LAB,
)

clf_wild, wild_test_top1, wild_test_top5 = train_mlp_model(
    'Wild',
    train_wild_seqs, train_wild_labels,
    val_wild_seqs, val_wild_labels,
    test_wild_seqs, test_wild_labels,
    N_CLASSES_WILD,
)

In [ ]:
# Final summary

print(f"\n{'='*70}")
print(f"Summary — LRRo word classification ({STRATEGY})")
print(f"{'='*70}")
print(f"  Encoder: VTP (frozen) + VSR transformer encoder ({MODEL_REPO})")
print(f"  Classifier: AttentionPooling + MLP")
print()
print(f"  {'Method':<40s} {'Lab T1':>8s} {'Lab T5':>8s} {'Wild T1':>9s} {'Wild T5':>9s}")
print(f"  {'-' * 70}")
print(f"  {'LRRo baseline (VGG-M / Inception)':<40s} {'71.0%':>8s} {'92.0%':>8s} {'33.0%':>9s} {'62.0%':>9s}")
print(f"  {'Ours (Attn+MLP, ' + STRATEGY + ')':<40s} {lab_test_top1:>7.1f}% {lab_test_top5:>7.1f}% {wild_test_top1:>8.1f}% {wild_test_top5:>8.1f}%")
print()
print(f"  Saved checkpoints in {OUTPUT_DIR}/:")
print(f"    - best_lab_clf.pt   ({N_CLASSES_LAB} classes)")
print(f"    - best_wild_clf.pt  ({N_CLASSES_WILD} classes)")